Notebook to set up the first pipe line for a speech-to-speech translation. 

In [7]:
from transformers import pipeline
import torch 

In [8]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.cuda.is_available()

True

In [9]:
classifier = pipeline(
    "audio-classification", model="MIT/ast-finetuned-speech-commands-v2", device=device
)

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 12099.53it/s]


In [10]:
classifier.model.config.id2label

{0: 'backward',
 1: 'follow',
 2: 'five',
 3: 'bed',
 4: 'zero',
 5: 'on',
 6: 'learn',
 7: 'two',
 8: 'house',
 9: 'tree',
 10: 'dog',
 11: 'stop',
 12: 'seven',
 13: 'eight',
 14: 'down',
 15: 'six',
 16: 'forward',
 17: 'cat',
 18: 'right',
 19: 'visual',
 20: 'four',
 21: 'wow',
 22: 'no',
 23: 'nine',
 24: 'off',
 25: 'three',
 26: 'left',
 27: 'marvin',
 28: 'yes',
 29: 'up',
 30: 'sheila',
 31: 'happy',
 32: 'bird',
 33: 'go',
 34: 'one'}

In [30]:
from transformers.pipelines.audio_utils import ffmpeg_microphone_live


def launch_fn(
    wake_word="marvin",
    prob_threshold=0.5,
    chunk_length_s=2.0,
    stream_chunk_s=0.25,
    debug=False,
):
    if wake_word not in classifier.model.config.label2id.keys():
        raise ValueError(
            f"Wake word {wake_word} not in set of valid class labels, pick a wake word in the set {classifier.model.config.label2id.keys()}."
        )

    sampling_rate = classifier.feature_extractor.sampling_rate

    mic = ffmpeg_microphone_live(
        sampling_rate=sampling_rate,
        chunk_length_s=chunk_length_s,
        stream_chunk_s=stream_chunk_s,
    )


    for chunk in mic:
        # Ignore the 0.25 s temporary pieces.
        if chunk["partial"]:
            continue

        # This will be approximately 2 seconds at 16 kHz.
        print("Classifying chunk:", chunk["raw"].shape)

        prediction = classifier(chunk)[0]

        if debug:
            print(prediction)

        if (
            prediction["label"] == wake_word
            and prediction["score"] > prob_threshold
        ):
            print("Wake word detected.")
            return True

In [31]:
launch_fn(debug=True)

Using microphone: Headset Microphone (2- DualSense Wireless Controller)
Classifying chunk: (32000,)
{'score': 0.10085275769233704, 'label': 'zero'}
Classifying chunk: (32000,)
{'score': 0.08333943039178848, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.48295485973358154, 'label': 'three'}
Classifying chunk: (32000,)
{'score': 0.7748754620552063, 'label': 'marvin'}
Wake word detected.


True